### Load the libraries

In [1]:
### Load the libraries 
import os, sys
import warnings
warnings.filterwarnings('ignore')
import pandas as pd 
import numpy as np 
import anndata as ad 
import scanpy as sc

<i><b> Set the home directory </b></i>

In [2]:
env_path = "../.env"

In [3]:
# call paths from .env
if os.path.exists(env_path):
    with open(env_path) as f:
        os.environ.update(
            line.strip().split("=", 1) for line in f
            if line.strip() and not line.startswith("#")
        )

data_dir = os.getenv("DATA_DIR")
ensembl_dir = os.getenv("ENSEMBL_DIR")
raw_adata = os.getenv("RAW_ANNDATA_DIR")

### Assembly - Linnarsson_mm_dg_GSE104323

In [4]:
# Load the expression data 
expression_linnarsson_mm_dg = pd.read_csv(data_dir + '/Linnarsson_mm_dg_GSE104323/GSE104323_10X_expression_data_V2.tab', 
                                          delimiter='\t' ,index_col=0)

In [5]:
# Load the metadata file
metadata_linnarsson_mm_dg = pd.read_csv(data_dir + '/Linnarsson_mm_dg_GSE104323/GSE104323_metadata_barcodes_24185cells.txt', 
                                        index_col=0, delimiter='\t')

metadata_linnarsson_mm_dg = metadata_linnarsson_mm_dg.dropna(how='any')

In [6]:
# Reset the index
metadata_linnarsson_mm_dg = metadata_linnarsson_mm_dg.reset_index()
# Build the initial dataframe
expression_linnarsson_mm_dg = expression_linnarsson_mm_dg.T
adata_linnarsson_mm_dg = ad.AnnData(expression_linnarsson_mm_dg)
# Add var_names and obs_names
adata_linnarsson_mm_dg.obs_names = metadata_linnarsson_mm_dg['Sample name (24185 single cells)']
adata_linnarsson_mm_dg.var_names = expression_linnarsson_mm_dg.T.index

In [7]:
# Add the metadata
adata_linnarsson_mm_dg.obs = metadata_linnarsson_mm_dg
adata_linnarsson_mm_dg

AnnData object with n_obs × n_vars = 24185 × 27933
    obs: 'Sample name (24185 single cells)', 'source name', 'organism', 'characteristics: strain', 'characteristics: age', 'characteristics: sex of pooled animals', 'characteristics: cell cluster', 'molecule', 'SRR run accession', 'raw file (original file name)', 'UMI_CellularBarcode'

In [8]:
# Save the anndata
adata_linnarsson_mm_dg.write_h5ad(raw_adata + 'Adata_linnarsson_mm_dg_raw.h5ad')

### Assembly - Zylka_mm_ncx_GSE123335

In [9]:
# Load the expression data
expression_zylka_mm_ncx = pd.read_csv(data_dir + '/Zylka_mm_ncx_GSE123335/GSE123335_E14_combined_matrix.txt', 
                                      delimiter='\t', index_col=0)

In [10]:
# Load the metadata
metadata_zylka_mm_ncx = pd.read_csv(data_dir + '/Zylka_mm_ncx_GSE123335/GSE123335_E14_combined_matrix_ClusterAnnotations.txt', 
                                    delimiter='\t', index_col=0)

In [11]:
# Print the sizes of the expression data and metadata
print(expression_zylka_mm_ncx.shape)
print(metadata_zylka_mm_ncx.shape)

# Expression matrix seems to have more cells (11069) than the metadata. 
# In the paper (https://www.nature.com/articles/s41467-018-08079-9) authors claim to have identified 10,931 cells, so I subsetted the expression matrix to those cellIDs. 

(21313, 11069)
(10931, 1)


In [12]:
# Reset the index
metadata_zylka_mm_ncx = metadata_zylka_mm_ncx.reset_index()
metadata_zylka_mm_ncx['CellID'] = metadata_zylka_mm_ncx['CellID'].str.replace('.', '-')
# Transpose
expression_zylka_mm_ncx = expression_zylka_mm_ncx.T
# Build the initial anndata object
adata_zylka_mm_ncx = ad.AnnData(expression_zylka_mm_ncx)

In [13]:
# Keep only cellIDs present in the metadata
metadata_barcodes = metadata_zylka_mm_ncx['CellID'].tolist()
adata_zylka_mm_ncx = adata_zylka_mm_ncx[adata_zylka_mm_ncx.obs_names.isin(metadata_barcodes)].copy()

In [14]:
# Add the metadata to the anndata object
adata_zylka_mm_ncx.obs = metadata_zylka_mm_ncx
adata_zylka_mm_ncx

AnnData object with n_obs × n_vars = 10931 × 21313
    obs: 'CellID', 'Cluster'

In [15]:
# Save the anndata
adata_zylka_mm_ncx.write_h5ad(raw_adata + 'Adata_zylka_mm_ncx_raw.h5ad')

### Assembly - Ruan_mm_ncx_GSE161690

In [16]:
# Load the expression data
from scipy.io import mmread
expression_ruan_mm_ncx = mmread(data_dir + '/Ruan_mm_ncx_GSE161690/GSE161690_count.mtx')
expression_ruan_mm_ncx_dense = expression_ruan_mm_ncx.toarray()

In [17]:
# Load the metadata
metadata_ruan_mm_ncx = pd.read_csv(data_dir + '/Ruan_mm_ncx_GSE161690/GSE161690_cell_info.csv', delimiter=',', index_col=0)

In [18]:
# Load the gene names
gene_names_ruan_mm_ncx = pd.read_csv(data_dir + '/Ruan_mm_ncx_GSE161690/GSE161690_gene_names.csv', delimiter=',', index_col=0)

In [19]:
# Assemble the anndata object
adata_ruan_mm_ncx = ad.AnnData(
    X = expression_ruan_mm_ncx_dense.T, 
    obs = metadata_ruan_mm_ncx, 
    var = pd.DataFrame(index=gene_names_ruan_mm_ncx)
)

adata_ruan_mm_ncx

AnnData object with n_obs × n_vars = 10261 × 21862
    obs: 'cellid', 'label'

In [20]:
# Save the anndata
adata_ruan_mm_ncx.write_h5ad(raw_adata + 'Adata_ruan_mm_ncx_raw.h5ad')